# Q# Resource Estimator Python Interface
The resource estimator can now be imported into Python as a module using the functionality provided by [PyO3](https://pyo3.rs/v0.27.1/getting-started.html). To set it up, first install maturin in your virtual environment:

`pip install maturin`

Then build and install the module locally by running:

`run maturin develop`

After this, you should be good to go and you can import the resource estimator directly from Python.

# Functionality
The python interface offers two main functionalities:
- The first option is to input a tuple ```(#Qubits, #CX, #CCX)``` for which you want to estimate the physical resources. These values can be obtained for example from Qualtran so this provides a seemless interface between the two tools.
- The second option is to provide a Q# file which specificies a concrete algorithm for which you want to estimate the physical resources. The tool will then calculate ```(#Qubits, #CX, #CCX)``` from this Q# sharp file and perform the resource estimation from there.

In the following I demonstrate these functionalities

In [15]:
# Import the Rust functionality as a Python module
import qsharp_alice_bob_resource_estimator as qre

# Qualtran

In [16]:
from qualtran_primitives.construction_helpers import ECCInstance, analyze_ecc_private_key_circuit

In [ ]:
cfg = ECCInstance(
    n=256,
    p=2**256 - 2**32 - 977,
    wa=16,
    wm=16,
    Gx=55066263022277343669578718895168534326250603453777594175500187360389116729240,
    Gy=32670510020758816978083085130507043184471273380659243275938904335757337482424,
    scalar=2026,
)

results = analyze_ecc_private_key_circuit(cfg)

gc = results["gate_cost"]


#### Counts totals:
 - `And`: 378940576
 - `And†`: 378875296
 - `ArbitraryClifford(n=256)`: 2
 - `CNOT`: 1153566112
 - `C[CNOT]`: 34030112
 - `C[PhaseGradientUnitary]`: 512
 - `C[X]`: 67689184
 - `H`: 512
 - `MeasureZ`: 1434112
 - `Toffoli`: 216186720
 - `TwoBitCSwap`: 83820544
 - `X`: 34063424
 - `times2`: 65536
 - `|+>`: 512

In [ ]:
nb_qbits = results["qubits"]
num_cx = int(results["num_cx"])
num_ccx = int(results["num_ccx"])


2324
1338222256
489477552


In [19]:
elliptic_curve_logic, _ = qre.estimate_resources_struct(nb_qbits,num_cx,num_ccx, frontier=False,error_total=None,error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0))

print("\n=== Logical_estimates example ===")
print(elliptic_curve_logic)


=== Logical_estimates example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    126,988
runtime:             16.43 hrs
total error:         0.13965
─────────────────────────────
code distance:       15 (|α|² = 20.00)
#factories:          43
factories distance:  9 (|α|² = 12.83)
factory fraction:    2.88%
─────────────────────────────



# ECC example

One example that is directly built into the resource estimator is the example of computing elliptic curve logarithms following the considerations in https://arxiv.org/abs/2302.06639.

In [20]:
print(qre.elliptic_curve_estimate_struct.__doc__)

Structured variant of the ECC example that returns typed estimate objects.

# Arguments
- `bit_size` — ECC modulus bit size.
- `window_size` — Window size used by the example algorithm.
- `frontier` — If `true`, also return a list representing the frontier.

# Returns
A tuple:
1. `EstimatesPy` — single best estimate,
2. `Vec<EstimatesPy>` — optional frontier (empty if `frontier == false`).

# Errors
Propagates example execution or estimation errors as Python `RuntimeError`s.


In [21]:
elliptic_curve, _ = qre.elliptic_curve_estimate_struct(256, 18, True)
print("\n=== Elliptic curve cryptography example ===")
print(elliptic_curve)


=== Elliptic curve cryptography example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    115,203
runtime:             7.57 hrs
total error:         0.23104
─────────────────────────────
code distance:       13 (|α|² = 19.00)
#factories:          61
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.50%
─────────────────────────────



The parameters can be easily accessed as class variables as seen below 

In [22]:
code_distance = elliptic_curve.code_distance
print(f"Code distance: {code_distance}")
# Same for other parameters

Code distance: 13


## Estimate resources from (#Qubits, #CX, #CCX)
The same example as above can be run in an equivalent way by using the more general functionality of passing ```(#Qubits, #CX, #CCX)``` as input.

In [23]:
print(qre.estimate_resources_struct.__doc__)

Estimate resources from explicit logical counts and return typed results,
optionally including a frontier of trade-offs.

# Arguments
- `qubits` — Logical (algorithm) qubit count.
- `cx` — Logical CX-equivalent two-qubit gate count.
- `ccx` — Logical CCX (Toffoli) gate count.
- `frontier` — If `true`, compute and return the frontier as structured objects.
- `error_total` — Overall error target; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` for an explicit split.

# Returns
A tuple:
1. `EstimatesPy` — single best estimate,
2. `Vec<EstimatesPy>` — frontier (empty if `frontier == false`).

# Errors
Propagates errors from the physical resource estimator.


In [35]:
import math

bit_size = 256
window_size = 18

qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (p. 21, app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)

elliptic_curve_logic, _ = qre.estimate_resources_struct(qubit_count,cx_count,ccx_count, frontier=False,error_total=None,error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0))

In [25]:
print("\n=== Logical_estimates example ===")
print(elliptic_curve_logic)


=== Logical_estimates example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    115,203
runtime:             7.57 hrs
total error:         0.23104
─────────────────────────────
code distance:       13 (|α|² = 19.00)
#factories:          61
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.50%
─────────────────────────────



## Estimate Resources from Q# File
The last option is to specify an algorithm in a Q# file and pass it as an input to the resource estimator.

In [26]:
print(qre.estimate_file_struct.__doc__)

Estimate resources from a Q# file and return both the best estimate and, optionally,
a frontier of Pareto-optimal trade-offs, together with the parsed logical counts.

# Arguments
- `filename` — Path to a Q# source file to be parsed and interpreted for counts.
- `frontier` — If `true`, also compute a frontier of estimates (e.g., different distances/α).
- `error_total` — Overall error target `p_total`; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` if an explicit split is desired.

# Returns
A 3-tuple:
1. `EstimatesPy` — the single best estimate,
2. `Vec<EstimatesPy>` — optionally, the frontier (empty if `frontier == false`),
3. `LogicalCountsPy` — Python snapshot of the logical counts extracted from `filename`.

# Errors
- I/O or parsing failures when loading the Q# file,
- Failures during resource estimation.



In [27]:
filename = "qsharp/Adder.qs"
single_qsharp, frontier, counts = qre.estimate_file_struct(
    filename, 
    frontier=True, 
    error_total=None, 
    error_budget=(0.0005, 0.0005, 0.0)
)
print("\n=== Q# example ===")
print(single_qsharp)


=== Q# example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    13,419
runtime:             0.00 hrs
total error:         0.00019
─────────────────────────────
code distance:       9 (|α|² = 14.00)
#factories:          2
factories distance:  5 (|α|² = 8.18)
factory fraction:    0.67%
─────────────────────────────

